# Wish Generation LLM Evaluation

This notebook evaluates different LLM models for personalized wish generation.

## Metrics Tracked
- **latency_ms**: Time per prompt
- **tokens_in / tokens_out**: Inference cost basis
- **quality_score**: LLM-as-a-judge focusing on:
  - Warmth and personalization
  - Age-appropriateness
  - Safety (no hallucinated dangerous/offensive content)
- **monthly_estimate**: Cost for 2B requests

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd() / "prototype" / "src"))

from src.evaluation import (
    evaluate_quality_with_judge,
    setup_mlflow,
)
from src.generators import generate_wish
from src.model_config import get_judge_client, get_models_for_evaluation
from src.prompts import (
    generate_wish_quality_judge_prompt,
)
from src.test_samples import get_test_profiles, get_test_recommendations_for_wish
from src.utils import load_env_from_repo_root

# Load .env file from repository root
load_env_from_repo_root()

In [2]:
setup_mlflow("wish_generation_eval")

MLflow tracking URI: https://public-tracking-e00-q0ycj5wbge9njs0-p4e20s5hwjds743-mlflow.gw.msp.eu-north1.nebius.cloud
MLflow experiment: <Experiment: artifact_location='mlflow-artifacts:/3', creation_time=1765819679942, experiment_id='3', last_update_time=1765819679942, lifecycle_stage='active', name='wish_generation_eval', tags={}>


In [3]:
test_profiles = get_test_profiles()
test_recommendations = get_test_recommendations_for_wish()

test_recommendations

[GiftRecommendation(id='71b65279-e8f0-42df-8ee4-554499c53acf', kid_id='75c7f06a-9e27-417f-953e-510fd2870cec', gifts=['DIY doll making kit', 'Watercolor painting set', 'Storybook collection'], rationale="These gifts match Emma's creative interests.", model_version='test'),
 GiftRecommendation(id='d85baf7c-093f-491b-a9a3-6369d329df09', kid_id='152964da-9302-410f-9379-a0d11c694d2c', gifts=['Advanced LEGO Technic set', 'Educational coding game', 'Mountain bike'], rationale="These gifts align with Lucas's interests in building and technology.", model_version='test'),
 GiftRecommendation(id='4242d69b-5150-41d1-b1c4-2115f2318c4f', kid_id='9e06fea4-4f03-44d5-a697-38ece0b69d6c', gifts=['Princess dress-up set', 'Soft teddy bear', 'Fairy tale book'], rationale="These gifts are perfect for Sophia's age and interests.", model_version='test'),
 GiftRecommendation(id='9457a138-6358-47a2-811b-94098cc9ab49', kid_id='f6ef59ff-b127-4cd0-aac8-3badc5006a1b', gifts=['Professional basketball', 'Wireless head

In [4]:

models_to_evaluate = get_models_for_evaluation(model_names=[
    "gpt-4o-mini",
    "openai-gpt-3.5-turbo",
    "tf/gpt-oss-20b",
    "tf/DeepSeek-R1-0528",
    ])

In [5]:

judge_client = get_judge_client()

In [ ]:
import re
from pathlib import Path

import mlflow
import pandas as pd

test_profiles = get_test_profiles()
test_recommendations = get_test_recommendations_for_wish()

results = []
agg_results = []

# Enable automatic tracing for all OpenAI API calls.
MLFLOW_AVAILABLE = bool(mlflow.get_tracking_uri())
mlflow.autolog()

# Evaluate all configured models
for model_config in models_to_evaluate:
    model_name = model_config["name"]
    client = model_config["client"]
    print(f"Evaluating model: {model_name}")

    with mlflow.start_run(run_name=f"{model_name}"):
        parent_run_id = mlflow.active_run().info.run_id if MLFLOW_AVAILABLE else None
        per_calls = []

        for kid_profile, gift_recommendation in zip(test_profiles, test_recommendations):
            try:
                wish, metrics = generate_wish(client, kid_profile, gift_recommendation)

                quality_score = None
                quality_rationale = None
                if judge_client:
                    judge_prompt = generate_wish_quality_judge_prompt(kid_profile, wish.text)
                    quality_score, quality_rationale = evaluate_quality_with_judge(judge_client, judge_prompt)

                tokens_in = metrics.get("tokens_in", 0) or 0
                tokens_out = metrics.get("tokens_out", 0) or 0
                cost_in = round(tokens_in * model_config["cost_per_1m_tokens_in"], 1)
                cost_out = round(tokens_out * model_config["cost_per_1m_tokens_out"], 1)
                cost_total = round(cost_in + cost_out, 1)

                record = {
                    "model": model_name,
                    "kid_id": kid_profile.id,
                    "latency_ms": round(metrics.get("latency_ms", 0), 0),
                    "tokens_in": tokens_in,
                    "tokens_out": tokens_out,
                    "quality_score": quality_score,
                    "cost_in_1m": cost_in,
                    "cost_out_1m": cost_out,
                    "cost_total_1m": cost_total,
                }
                per_calls.append(record)
                results.append(record)

                if MLFLOW_AVAILABLE:
                    with mlflow.start_run(run_name=f"{kid_profile.id}", nested=True):
                        mlflow.log_metric("latency_ms", record["latency_ms"])
                        mlflow.log_metric("tokens_in", tokens_in)
                        mlflow.log_metric("tokens_out", tokens_out)
                        if quality_score is not None:
                            mlflow.log_metric("quality_score", quality_score)
                        mlflow.log_metric("cost_in_1m", cost_in)
                        mlflow.log_metric("cost_out_1m", cost_out)
                        mlflow.log_metric("cost_total_1m", cost_total)
                        mlflow.set_tags({"task": "wish_generation"})
                        mlflow.log_dict(
                            {
                                "wish": wish.text,
                                "quality_score": quality_score,
                                "quality_rationale": quality_rationale,
                            },
                            "wish_report.json"
                        )

            except Exception as e:
                err_rec = {
                    "model": model_name,
                    "kid_id": kid_profile.id,
                    "error": str(e),
                }
                per_calls.append(err_rec)
                results.append(err_rec)

        df_model = pd.DataFrame([r for r in per_calls if "error" not in r])
        if not df_model.empty:
            agg = {
                "model": model_name,
                "latency_ms": round(df_model["latency_ms"].mean(), 0),
                "tokens_in": df_model["tokens_in"].mean(),
                "tokens_out": df_model["tokens_out"].mean(),
                "quality_score": df_model["quality_score"].mean(),
                "cost_in_1m": round(df_model["cost_in_1m"].mean(), 1),
                "cost_out_1m": round(df_model["cost_out_1m"].mean(), 1),
                "cost_total_1m": round(df_model["cost_total_1m"].mean(), 1),
                "calls": len(df_model),
            }
            agg_results.append(agg)

            out_dir = Path("data/evaluation")
            out_dir.mkdir(parents=True, exist_ok=True)
            model_name_path = re.sub(r'[^\w\-]', '-', model_name)
            out_path = out_dir / f"03_wish_eval-{model_name_path}.csv"
            df_model.to_csv(out_path, index=False)
            print(f"Saved per-call results for {model_name} -> {out_path}")

            if MLFLOW_AVAILABLE:
                mlflow.log_metric("latency_ms", agg["latency_ms"])
                mlflow.log_metric("tokens_in", agg["tokens_in"])
                mlflow.log_metric("tokens_out", agg["tokens_out"])
                mlflow.log_metric("quality_score", agg["quality_score"])
                mlflow.log_metric("cost_in_1m", agg["cost_in_1m"])
                mlflow.log_metric("cost_out_1m", agg["cost_out_1m"])
                mlflow.log_metric("cost_total_1m", agg["cost_total_1m"])
                mlflow.log_metric("calls", agg["calls"])
                mlflow.set_tags({"task": "wish_generation"})
        else:
            print(f"No successful calls for model {model_name}")

Evaluating model: gpt-4o-mini
🏃 View run e23e861b-cdf0-4a53-8713-050c35fc6adc at: https://public-tracking-e00-q0ycj5wbge9njs0-p4e20s5hwjds743-mlflow.gw.msp.eu-north1.nebius.cloud/#/experiments/3/runs/2f3e53b0fdfe4e3684f7ea8bc51c25c4
🧪 View experiment at: https://public-tracking-e00-q0ycj5wbge9njs0-p4e20s5hwjds743-mlflow.gw.msp.eu-north1.nebius.cloud/#/experiments/3
🏃 View run 5c11fa7c-0aad-4e3b-a50c-9e851ba85cee at: https://public-tracking-e00-q0ycj5wbge9njs0-p4e20s5hwjds743-mlflow.gw.msp.eu-north1.nebius.cloud/#/experiments/3/runs/89d37391502b4c9abfce2cc547289664
🧪 View experiment at: https://public-tracking-e00-q0ycj5wbge9njs0-p4e20s5hwjds743-mlflow.gw.msp.eu-north1.nebius.cloud/#/experiments/3
🏃 View run 4dd60d63-64d5-45d9-99b5-357188c2e858 at: https://public-tracking-e00-q0ycj5wbge9njs0-p4e20s5hwjds743-mlflow.gw.msp.eu-north1.nebius.cloud/#/experiments/3/runs/0f20a71af77345559c85200ad1fb8a7c
🧪 View experiment at: https://public-tracking-e00-q0ycj5wbge9njs0-p4e20s5hwjds743-mlflow.gw

In [11]:
# Final aggregated summary
df_results = pd.DataFrame(results)
df_agg = pd.DataFrame(agg_results)
print("Per-model aggregate summary:")
print(df_agg.round(3))

Per-model aggregate summary:
                  model  latency_ms  tokens_in  tokens_out  quality_score  \
0           gpt-4o-mini      3056.0     175.50        77.5          0.900   
1  openai-gpt-3.5-turbo      1082.0     175.75        63.0          0.675   
2        tf/gpt-oss-20b      1154.0     237.50       200.0          0.000   
3   tf/DeepSeek-R1-0528      3031.0     175.00        65.5          0.900   

   cost_in_1m  cost_out_1m  cost_total_1m  calls  
0        26.3         46.5           72.8      4  
1       263.6        126.0          389.6      4  
2        35.6        120.0          155.6      4  
3       140.0        157.2          297.2      4  
